# Heart Disease — Model Training Pipeline
In this notebook, we train 6 machine learning models to detect Heart Disease:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine (SVM)
5. K-Nearest Neighbors (KNN)
6. Gradient Boosting

In [1]:
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Load Data and Split Features & Labels
Split into 80% training data and 20% testing data.

In [2]:
# Load cleaned heart disease dataset
df = pd.read_csv(os.path.join("processed_data", "cleaned_heart_disease.csv"))

# Separate features (X) and target (y)
X = df.drop(columns=['target'])
y = df['target']

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

X_train shape: (1600, 13), y_train shape: (1600,)
X_test shape:  (400, 13), y_test shape:  (400,)


## 2. Feature Scaling (StandardScaler)
Standardize features so all variables have mean = 0 and variance = 1.
Save the fitted scaler to `saved_models/heart_scaler.pkl`.

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler
joblib.dump(scaler, os.path.join(MODEL_DIR, "heart_scaler.pkl"))
print("[OK] Saved scaler to saved_models/heart_scaler.pkl")

[OK] Saved scaler to saved_models/heart_scaler.pkl


## 3. Train and Save Models
- Distance and linear models (Logistic Regression, SVM, KNN) use **scaled features**.
- Tree-based models (Decision Tree, Random Forest, Gradient Boosting) use **raw features**.

In [4]:
# Define dictionary of models: {filename: (model_instance, use_scaled_data)}
models = {
    'heart_logistic_regression.pkl': (LogisticRegression(max_iter=1000, random_state=42), True),
    'heart_decision_tree.pkl': (DecisionTreeClassifier(max_depth=5, random_state=42), False),
    'heart_random_forest.pkl': (RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42), False),
    'heart_rbf_svm.pkl': (SVC(kernel='rbf', probability=True, random_state=42), True),
    'heart_knn.pkl': (KNeighborsClassifier(n_neighbors=7), True),
    'heart_gradient_boosting.pkl': (GradientBoostingClassifier(random_state=42), False)
}

# Train and save each model
for filename, (model, use_scaled) in models.items():
    X_train_data = X_train_scaled if use_scaled else X_train
    model.fit(X_train_data, y_train)
    save_path = os.path.join(MODEL_DIR, filename)
    joblib.dump(model, save_path)
    print(f"[OK] Trained & Saved: {filename}")

[OK] Trained & Saved: heart_logistic_regression.pkl
[OK] Trained & Saved: heart_decision_tree.pkl
[OK] Trained & Saved: heart_random_forest.pkl


[OK] Trained & Saved: heart_rbf_svm.pkl
[OK] Trained & Saved: heart_knn.pkl


[OK] Trained & Saved: heart_gradient_boosting.pkl


## 4. Verification
Check all files saved in `saved_models/`.

In [5]:
print("Heart disease model files:")
for f in sorted(os.listdir(MODEL_DIR)):
    if f.startswith("heart_"):
        print(" -", f)

Heart disease model files:
 - heart_decision_tree.pkl
 - heart_gradient_boosting.pkl
 - heart_knn.pkl
 - heart_logistic_regression.pkl
 - heart_random_forest.pkl
 - heart_rbf_svm.pkl
 - heart_scaler.pkl
